In [1]:
import pandas as pd
import polars as pl

## Key Conceptual Differences

Before diving into specific operations, let's understand the fundamental differences in how pandas and Polars approach DataFrame manipulation.

### No Index in Polars

One of the most significant differences is that Polars has no concept of an index, while in pandas, every DataFrame has an index that plays a central role in many operations.

In [2]:
# pandas: Index is central to DataFrame identity
df_pd = pd.DataFrame({'A': [101, 102, 103], 'B': ["red", "green", "blue"], 'C': [2, 5, 7]})
df_pd.set_index('A', inplace=True)  # 'A' becomes special
df_pd

,B,C
A,,
101,red,2
102,green,5
103,blue,7


In [3]:
df_pd.loc[102]  # Access by index value

B    green
C        5
Name: 102, dtype: object

In [4]:
# Polars: All columns are equal
df_pl = pl.DataFrame({'A': [101, 102, 103], 'B': ["red", "green", "blue"]})
# No special index - all columns have the same status
df_pl

A,B
i64,str
101,"""red"""
102,"""green"""
103,"""blue"""


In [5]:
df_pl.filter(pl.col('A') == 102)  # Filter by condition

A,B
i64,str
102,"""green"""


### Immutability vs. In-Place Operations

pandas allows and sometimes encourages in-place modifications, while Polars generally follows an immutable pattern:

In [6]:
# pandas: In-place modification with inplace=True
df_pd = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df_pd['C'] = df_pd['B'] * 2  # Add column in-place
df_pd.drop('B', axis=1, inplace=True)  # Remove column in-place

# Polars: Operations return new DataFrames
df_pl = pl.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df_pl = df_pl.with_columns((pl.col('B') * 2).alias('C'))  # Add column
df_pl = df_pl.drop('B')  # Remove column

### Expression API vs. Apply/Lambda

Perhaps the most fundamental difference for everyday coding is how transformations are specified. pandas often relies on Python functions (via `.apply()` or lambdas), while Polars emphasizes declarative expressions:

In [7]:
# pandas: Using apply with a lambda
df_pd = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 6, 8]})
df_pd['C'] = df_pd.apply(lambda row: row['A'] * 2 if row['B'] > 5 else row['A'], axis=1)

# Polars: Using expressions
df_pl = pl.DataFrame({'A': [1, 2, 3], 'B': [4, 6, 8]})
df_pl = df_pl.with_columns(
    pl.when(pl.col('B') > 5)
      .then(pl.col('A') * 2)
      .otherwise(pl.col('A'))
      .alias('C')
)

### Eager vs. Lazy Evaluation

While pandas operates almost exclusively in eager mode (operations execute immediately), Polars offers both eager and lazy modes:

In [8]:
# pandas: Always eager
# df_pd = pd.read_csv("data.csv")
# filtered = df_pd[df_pd['value'] > 0]  # Executes immediately
# result = filtered.groupby('category').sum()  # Executes immediately

# Polars: Eager mode
# df_pl = pl.read_csv("data.csv")
# filtered = df_pl.filter(pl.col('value') > 0)  # Executes immediately
# result = filtered.group_by('category').agg(pl.sum('value'))  # Executes immediately

# Polars: Lazy mode
# lazy_df = pl.scan_csv("data.csv")
# query = (
#     lazy_df.filter(pl.col('value') > 0)
#     .group_by('category')
#     .agg(pl.col('value').sum())
# )
# No execution yet - just building a query plan
# result = query.collect()  # Now executes with optimization

### Type Strictness

Polars is stricter about types and schema consistency:

In [9]:
# pandas: Relatively permissive about types
df_pd = pd.DataFrame({'A': [1, 2, 'three']})  # Converts to object dtype

# Polars: Stricter about types
try:
    df_pl = pl.DataFrame({'A': [1, 2, 'three']})
except Exception as e:
    print(f"Polars requires consistent types: {e}")

Polars requires consistent types: unexpected value while building Series of type Int64; found value of type String: "three"

Hint: Try setting `strict=False` to allow passing data with mixed types.


## Common Operations: pandas vs. Polars

Now let's look at side-by-side comparisons of common operations in both libraries.

### Creating DataFrames

In [10]:
# pandas
import pandas as pd
df_pd = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['a', 'b', 'c'],
    'C': [True, False, True]
})

# Polars
import polars as pl
df_pl = pl.DataFrame({
    'A': [1, 2, 3],
    'B': ['a', 'b', 'c'],
    'C': [True, False, True]
})

### Selecting Columns

In [11]:
# pandas
sub_pd = df_pd[['A', 'B']]
# or
sub_pd = df_pd.loc[:, ['A', 'B']]

# Polars
sub_pl = df_pl.select(['A', 'B'])
# or with expressions
sub_pl = df_pl.select(pl.col(['A', 'B']))

### Filtering Rows

In [12]:
# pandas
filtered_pd = df_pd[df_pd['A'] > 1]
# or
filtered_pd = df_pd.loc[df_pd['A'] > 1]
# or
filtered_pd = df_pd.query("A > 1")

# Polars
filtered_pl = df_pl.filter(pl.col('A') > 1)

### Adding/Modifying Columns

In [13]:
# pandas
df_pd['D'] = df_pd['A'] * 2
# or
df_pd = df_pd.assign(D=df_pd['A'] * 2)

# Polars
df_pl = df_pl.with_columns((pl.col('A') * 2).alias('D'))
# or with multiple columns
df_pl = df_pl.with_columns([
    (pl.col('A') * 2).alias('D'),
    (pl.col('A') + 10).alias('E') # Modified example slightly for valid types
])

### GroupBy Operations

In [14]:
# pandas
grouped_pd = df_pd.groupby('B').agg({'A': ['sum', 'mean']})
# Reset index to make 'B' a regular column again
grouped_pd = grouped_pd.reset_index()

# Polars
grouped_pl = df_pl.group_by('B').agg([
    pl.col('A').sum().alias('A_sum'),
    pl.col('A').mean().alias('A_mean')
])

### Joins

In [15]:
# pandas
left_pd = pd.DataFrame({'key': ['A', 'B', 'C'], 'value': [1, 2, 3]})
right_pd = pd.DataFrame({'key': ['A', 'B', 'D'], 'other': [4, 5, 6]})
joined_pd = pd.merge(left_pd, right_pd, on='key', how='left')

# Polars
left_pl = pl.DataFrame({'key': ['A', 'B', 'C'], 'value': [1, 2, 3]})
right_pl = pl.DataFrame({'key': ['A', 'B', 'D'], 'other': [4, 5, 6]})
joined_pl = left_pl.join(right_pl, on='key', how='left')

### Handling Missing Values

In [16]:
# pandas
df_pd = df_pd.dropna()  # Drop rows with any NaN
df_pd = df_pd.fillna(0)  # Fill NaN with 0

# Polars
df_pl = df_pl.drop_nulls()  # Drop rows with any null
df_pl = df_pl.fill_null(0)  # Fill nulls with 0

### Sorting

In [17]:
# pandas
sorted_pd = df_pd.sort_values(by=['A', 'B'], ascending=[True, False])

# Polars
sorted_pl = df_pl.sort(['A', 'B'], descending=[False, True])

## Expression API: The Heart of the Difference

The most profound difference for day-to-day coding is Polars' expression API versus pandas' more Python-centric approach. Let's explore this with more examples.

### Conditional Logic

In [18]:
# pandas: Using numpy where or apply
import numpy as np
df_pd = pd.DataFrame({'value': [10, 60, 120]})
df_pd['category'] = np.where(df_pd['value'] > 100, 'high', 'low')

# More complex conditions often use apply
df_pd['category'] = df_pd.apply(
    lambda row: 'high' if row['value'] > 100 else 'medium' if row['value'] > 50 else 'low',
    axis=1
)

# Polars: Using when/then/otherwise expressions
df_pl = pl.DataFrame({'value': [10, 60, 120]})
df_pl = df_pl.with_columns(
    pl.when(pl.col('value') > 100)
      .then(pl.lit('high'))
      .otherwise(pl.lit('low'))
      .alias('category')
)

# More complex conditions chain naturally
df_pl = df_pl.with_columns(
    pl.when(pl.col('value') > 100)
      .then(pl.lit('high'))
      .when(pl.col('value') > 50)
      .then(pl.lit('medium'))
      .otherwise(pl.lit('low'))
      .alias('category')
)

### String Operations

In [19]:
# pandas: Using the .str accessor
df_pd = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie']})
df_pd['name_upper'] = df_pd['name'].str.upper()
df_pd['contains_a'] = df_pd['name'].str.contains('a')

# Polars: Using string expressions
df_pl = pl.DataFrame({'name': ['Alice', 'Bob', 'Charlie']})
df_pl = df_pl.with_columns([
    pl.col('name').str.to_uppercase().alias('name_upper'),
    pl.col('name').str.contains('a').alias('contains_a')
])

### Window Functions

In [20]:
# pandas
df_pd = pd.DataFrame({'group': ['A', 'A', 'B', 'B'], 'value': [1, 2, 3, 4]})
df_pd['cumsum'] = df_pd.groupby('group')['value'].transform('cumsum')
df_pd['rank'] = df_pd.groupby('group')['value'].rank()

# Polars
df_pl = pl.DataFrame({'group': ['A', 'A', 'B', 'B'], 'value': [1, 2, 3, 4]})
df_pl = df_pl.with_columns([
    pl.col('value').cum_sum().over('group').alias('cumsum'),
    pl.col('value').rank().over('group').alias('rank')
])

## Migration Strategies

Given the differences outlined above, how should you approach migrating from pandas to Polars? Here are some practical strategies:

### 1. Incremental Adoption

You don't need to migrate everything at once. Polars and pandas interoperate well:

In [21]:
# Convert pandas DataFrame to Polars
df_pl = pl.from_pandas(df_pd)

# Convert Polars DataFrame to pandas
df_pd_back = df_pl.to_pandas()

### 2. Start with New Projects

For new data processing projects, consider starting with Polars from the beginning. This avoids the complexity of migration and lets you design around Polars' strengths from the start.

### 3. Focus on Performance Bottlenecks

If you're migrating an existing codebase, focus first on the parts where performance matters most:
- Large dataset operations
- Operations that currently use `apply()`
- Complex aggregations and joins
- Operations in critical processing paths

### 4. Rewrite Common Patterns

Look for these common pandas patterns that should be rewritten in Polars:

- **`df[df['column'] > value]`** → **`df.filter(pl.col('column') > value)`**
- **`df.loc[:, ['A', 'B']]`** → **`df.select(['A', 'B'])`**
- **`df['new_col'] = df['A'] + df['B']`** → **`df = df.with_columns((pl.col('A') + pl.col('B')).alias('new_col'))`**
- **`df.apply(lambda x: ...)`** → Use Polars expressions instead
- **`df.groupby('A').agg(...).reset_index()`** → **`df.group_by('A').agg(...)`**

### 5. Leverage Lazy Evaluation

For complex data pipelines, adopt lazy evaluation to get the full performance benefits:

In [22]:
# Instead of eager operations
# df = pl.read_csv("data.csv")
# result = (df
#     .filter(pl.col('value') > 0)
#     .group_by('category')
#     .agg(pl.col('value').sum())
# )

# Use lazy evaluation
# result = (pl.scan_csv("data.csv")
#     .filter(pl.col('value') > 0)
#     .group_by('category')
#     .agg(pl.col('value').sum())
#     .collect()
# )

## Conclusion: Is Migration Worth It?

After reviewing the differences and migration strategies, the question remains: is migrating from pandas to Polars worth the effort?

The answer depends on your specific needs, but here are some considerations:

### When Migration Makes Sense

- Your workflows involve large datasets (millions of rows)
- Performance is a bottleneck in your current pandas code
- You're doing complex analytical operations (joins, groupby, window functions)
- You value memory efficiency
- You're starting new projects and can design around Polars from the beginning

### When You Might Want to Stick with pandas

- Your current pandas code performs adequately for your needs
- You rely heavily on pandas' ecosystem integrations
- You have a large codebase that would be costly to migrate
- Your team has deep pandas expertise and limited bandwidth to learn new tools
- You need specialized features that Polars doesn't yet support

Remember that this isn't an all-or-nothing choice. The interoperability between pandas and Polars allows for incremental adoption, using each tool where it makes the most sense.